In [0]:
from pyspark.sql.functions import col, monotonically_increasing_id, to_date

In [0]:
print("Construyendo la Tabla de Hechos (Gold Layer - Fact Sales)...")

Construyendo la Tabla de Hechos (Gold Layer - Fact Sales)...


In [0]:
# 1. Cargar las tablas de la capa Plata y las Dimensiones de la capa Oro
df_integrated_sales = spark.table("workspace.default.silver_integrated_sales")
dim_product = spark.table("workspace.default.gold_dim_product")

In [0]:
# 2. Preparar las llaves foráneas
# Extraemos la fecha pura para que conecte con dim_date
df_sales_prepared = df_integrated_sales.withColumn("date_key", to_date(col("order_purchase_timestamp")))

In [0]:
# 3. Cruzar con las dimensiones para obtener las llaves subrogadas (Surrogate Keys)
fact_sales = df_sales_prepared.join(
    dim_product,
    df_sales_prepared.product_id == dim_product.product_business_key,
    "inner"
).select(
    col("order_id"),
    col("product_key"),           # Llave foránea hacia dim_product
    col("date_key"),              # Llave foránea hacia dim_date
    col("price"),                 # Métrica base
    col("freight_value"),         # Métrica base
    (col("price") + col("freight_value")).alias("total_item_revenue") # Métrica de negocio derivada
).withColumn("sales_key", monotonically_increasing_id()) # Llave primaria de la Fact Table

In [0]:
# 4. Guardar la tabla de hechos
fact_sales.write.format("delta").mode("overwrite").saveAsTable("workspace.default.gold_fact_sales")

In [0]:
print("✅ Capa Oro (Fact Table) generada exitosamente. ¡El Data Warehouse está completo!")

✅ Capa Oro (Fact Table) generada exitosamente. ¡El Data Warehouse está completo!
